# Feature Importance (SHAP)

## Set Up

Packages/Libraries

In [ ]:
import sys

sys.path.append("../")
from src.data_utils import get_data, get_models
from src.config import BASE_PATH
from src.feat_imp import get_shap_single_model
import joblib

Set globals

In [ ]:
# Data
DATA_DICT = {
    "base": get_data(is_nomo=False),
    "nomo": get_data(is_nomo=True),
}

# Shap backgrounds
BACKGROUNDS = joblib.load(BASE_PATH / "artifacts/shap_backgrounds.joblib")

# Models
model_dir = BASE_PATH / "artifacts/models/trained"
model_prefix_list = ["lgbm", "xgb", "knn", "svc", "nn", "stack"]

## Base models
model_dict = get_models(model_prefix_list, model_dir)
## Nomogram
model_dict.update(get_models(["lr"], model_dir))


FEAT_ORDER = [
    ##Pre-Op
    "AGE",
    "BMI",
    "SEX",
    "Diabetes",
    "ASA",
    "PRIOREX",
    "PRECT",
    ## Disease
    "RECUR",
    "SITE",
    "SIZE",
    "LYMPH",
    "STAGE",
    "DEFECT",
    "SECONDPRIMARY",
    ## Surg
    "LENGTH",
    "JEWER",
    "OSTEOTOMY",
    "PLATE",
    "FLAP",
    "TRANSFUS",
    "ISCHEMICTIME",
    "OPTIME",
    ## Immediate post-op
    "REOP",
    "LOHS",
    "POSTCT",
    "WOUNDINF",
    "HGB",
    "ALB",
    ## Long term post-op
    "EXPOSURE",
    "MEDUSED",
    "SURGUSED",
    "PLATETIME",
    "FOLLOWTIME",
    ## Misc
    "RADTIME",
]

# RUN SHAP

In [ ]:
result_path = BASE_PATH / "results/tables/SHAP.xlsx"
if result_path.exists():
    result_path.unlink()
jobs = []
for model_name, model in model_dict.items():
    print(f"Model: {model_name}...")
    ## =====================> SET UP DATA
    keyword = "nomo" if model_name == "lr" else "base"
    ## Background
    X_background = BACKGROUNDS[model_name]
    ## Explain set
    X_explain = DATA_DICT[keyword]["X"]

    assert (
        list(X_explain.columns) == BACKGROUNDS[f"{model_name}__columns"]
    ), "column order drift"
    ## Results
    get_shap_single_model(
        model=model,
        model_name=model_name,
        feat_order=FEAT_ORDER,
        explanation_vals=X_explain,
        background_vals=X_background,
        result_path=result_path,
    )